In [ ]:
from scripts.AbstractExperiments import ExperimentRunner, GenerateConfigs
from EnvironmentBuilder.SearchRescue.Rescue import Rescue
import EnvironmentBuilder.SearchRescue.Drawing as SearchRescueDraw
from scripts.TexTables import SaveDataFrameToTexTemplate
from copy import deepcopy
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Charter",
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})
def latex_config_name(name):
    return "$" + name.replace("^0", "^{0}") + "$"

In [ ]:
def GenerateTeamConfigs(teams_:list, name_prefix=None):
    configs = {}
    if name_prefix is None:
        name_prefix = f"{len(teams_)}_"
    # Act-Utilitarianism
    con = {"Theories": [["Add_Util", "Utility", 0]], "Considerations": []}
    for t in teams_:
        con["Considerations"].append([f"{t}:wellbeing", "Add_Util"])
    configs[f"{name_prefix}add"] = con

    # Balance
    con = {"Theories": [], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", f"{t}"])
    configs[f"{name_prefix}bal"] = con

    # Fairness
    con = {"Theories": [["Fair", "Fairness", 0]], "Considerations": []}
    for t in teams_:
        con["Considerations"].append([f"{t}:wellbeing", ["Fair"]])
    configs[f"{name_prefix}fair"] = con

    # Fairness + Others
    con = {"Theories": [["Fair", "Fairness", 0]], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", [f"{t}", "Fair"]])
    configs[f"{name_prefix}fair_bal"] = con

    # Fairness + Act-Utilitarianism
    con = {"Theories": [["Fair", "Fairness", 0]], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", [f"{t}", "Fair"]])
    configs[f"{len(teams_)}fair_AU"] = con

    # Rawls
    con = {"Theories":[["Rawls", "Maximin", 0]], "Considerations": []}
    for t in teams_:
        con["Considerations"].append([f"{t}:wellbeing", ["Rawls"]])
    configs[f"{name_prefix}rawl"] = con

    # Rawls + Others
    con = {"Theories": [["Rawls", "Maximin", 0]], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", [f"{t}", "Rawls"]])
    configs[f"{name_prefix}rawl_bal"] = con

    # Rawls + AU
    con = {"Theories": [["Rawls", "Maximin", 0], ["Add_Util", "Utility", 0]], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", ["Rawls", "Add_Util"]])
    configs[f"{name_prefix}rawl_AU"] = con

    return configs

def MakeTeamWeights(teams_):
    p = [2**i for i in range(0, len(teams_))]
    total = sum(p)
    w = [i / total for i in p]
    o = {}
    for t in teams_:
        for i, t in enumerate(teams_):
            o[f"{t}_hosp_success"] = w[i]
    return o

markers = {
    "Separate Theories": "o",
    "Act-Utilitarianism": "s",
    "Fairness": "^",
    "Rawls": "*",
    "Fair + balance": ".",
    "Rawls + balance": "x"
}
colours = {
    "Separate Theories": "blue",
    "Act-Utilitarianism": "black",
    "Fairness": "green",
    "Rawls": "red",
    "Fair + balance": "cyan",
    "Rawls + balance": "magenta",
}

In [ ]:
env_reps = 1
for horizon_ in range(3,7):
    for d in range(0,3):
        teams = ["red", "blue", "green"]
        odds = MakeTeamWeights(teams)
        configs += GenerateConfigs(inputConfigs=GenerateTeamConfigs(teams, name_prefix=f"{len(teams)}_h{horizon_}"), 
            defaultConfig={"Horizon": horizon_, "Teams":teams, "unknown_depth": d, "Odds":odds})

er = ExperimentRunner("Rescue", configs, MoralPlanner_Location="/../../")
er.buildEnvironments()

In [ ]:
er.runPlanner(envRepetitions=env_reps)
er.extractData(1, env_reps)

## Get community wellbeing data for each individual policy

In [ ]:
cols = [f"{t}:wellbeing" for t in teams]
worth_df = pd.DataFrame(er.GetWorthData(worth_tags=[f"{t}:wellbeing" for t in teams], envRepetitions=env_reps))
for t in cols:
    worth_df[t] = pd.to_numeric(worth_df[t])
worth_df = worth_df.drop(columns=["Conf_rep"])
worth_df['communities'] = worth_df['Config_name'].str[:1]
worth_df["Config_name"] = np.select(
    [
        worth_df["Config_name"].str.contains("fair_bal", case=False, na=False),
        worth_df["Config_name"].str.contains("fair_AU", case=False, na=False),
        worth_df["Config_name"].str.contains("rawl_bal", case=False, na=False),
        worth_df["Config_name"].str.contains("rawl_AU", case=False, na=False),
        worth_df["Config_name"].str.contains("add", case=False, na=False),
        worth_df["Config_name"].str.contains("bal", case=False, na=False),
        worth_df["Config_name"].str.contains("fair", case=False, na=False),
        worth_df["Config_name"].str.contains("rawl", case=False, na=False),
    ],
    [   
        "Fair + Separate",
        "Fair + AU",
        "Rawls + Separate",
        "Rawls + AU",
        "Act-Utilitarianism",
        "Separate Theories",
        "Fairness",
        "Rawls",
    ],
    default=worth_df["Config_name"]
)
worth_df

## Get min, max, average and std. dev between communities for each policy.

In [ ]:
policy_df = worth_df.copy()

# Detect the community wellbeing columns.
wellbeing_cols = [
    col for col in policy_df.columns
    if col.endswith(":wellbeing")
]

if not wellbeing_cols:
    raise ValueError("No columns ending with ':wellbeing' were found.")

# Ensure the wellbeing values are numeric.
policy_df[wellbeing_cols] = policy_df[wellbeing_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# NaN values are ignored. Thus, an experiment with two communities only uses
# 0:wellbeing and 1:wellbeing.
policy_df["Policy wellbeing minimum"] = policy_df[wellbeing_cols].min(
    axis=1,
    skipna=True
)

policy_df["Policy wellbeing maximum"] = policy_df[wellbeing_cols].max(
    axis=1,
    skipna=True
)

policy_df["Policy wellbeing average"] = policy_df[wellbeing_cols].mean(
    axis=1,
    skipna=True
)

# ddof=0 gives the population standard deviation.
policy_df["Policy wellbeing standard deviation"] = policy_df[
    wellbeing_cols
].std(
    axis=1,
    skipna=True,
    ddof=0
)

policy_df

## Average min, max, average and std. for horizon.

In [ ]:
policy_metric_cols = [
    "Policy wellbeing minimum",
    "Policy wellbeing maximum",
    "Policy wellbeing average",
    "Policy wellbeing standard deviation",
]

configuration_horizon_policy_table = (
    policy_df
    .groupby(
        ["Config_name", "Horizon", "communities"],
        observed=True,
        as_index=False,
    )[policy_metric_cols]
    .mean()
    .rename(
        columns={
            "Policy wellbeing minimum":
                "Average policy minimum",
            "Policy wellbeing maximum":
                "Average policy maximum",
            "Policy wellbeing average":
                "Average policy mean",
            "Policy wellbeing standard deviation":
                "Average policy standard deviation",
        }
    )
    .sort_values(["Horizon", "Config_name"])
)

configuration_horizon_policy_table.round(4)

## Average min, max, average and std. dev for configuration, from horizon averages

In [ ]:
policy_metric_cols = [
    "Policy wellbeing minimum",
    "Policy wellbeing maximum",
    "Policy wellbeing average",
    "Policy wellbeing standard deviation",
]

# The requested table: average each policy-level statistic across all policies
# generated by each configuration.
configuration_policy_table = (
    policy_df
    .groupby("Config_name", observed=True)[policy_metric_cols]
    .mean()
    .rename(
        columns={
            "Policy wellbeing minimum": "Average policy minimum",
            "Policy wellbeing maximum": "Average policy maximum",
            "Policy wellbeing average": "Average policy mean",
            "Policy wellbeing standard deviation":
                "Average policy standard deviation",
        }
    )
    .reset_index()
)

configuration_policy_table.round(4)

## Calcualte whiskers data from mean, min and max

In [ ]:
configuration_plot_statistics = (
    policy_df
    .groupby("Config_name", observed=True)[policy_metric_cols]
    .agg(["mean", "min", "max"])
)

configuration_plot_statistics.round(4)

In [ ]:
metric_labels = {
    "Policy wellbeing minimum": "Minimum",
    "Policy wellbeing maximum": "Maximum",
    "Policy wellbeing average": "Mean",
    "Policy wellbeing standard deviation": "Standard deviation",
}

# Select configurations and set their display order here.
configuration_order = [
    "Act-Utilitarianism",
    "Fairness",
    "Rawls",
    "Separate Theories",
    "Fair + Separate",
    "Rawls + Separate",
    "Fair + AU",
    "Rawls + AU",
]

# Optional: remove names that are not present in the statistics table.
configuration_order = [
    config
    for config in configuration_order
    if config in configuration_plot_statistics.index
]

x = np.arange(len(configuration_order))
number_of_metrics = len(metric_labels)
bar_width = 0.8 / number_of_metrics

figure_width = max(10, 1.6 * len(configuration_order))
fig, ax = plt.subplots(figsize=(figure_width, 6))

for metric_index, (metric, label) in enumerate(metric_labels.items()):
    means = (
        configuration_plot_statistics[(metric, "mean")]
        .reindex(configuration_order)
        .to_numpy()
    )

    positions = (
        x
        + metric_index * bar_width
        - ((number_of_metrics - 1) * bar_width / 2)
    )

    ax.bar(
        positions,
        means,
        width=bar_width,
        label=label,
    )

ax.set_xlabel("Configuration")
ax.set_ylabel("Community wellbeing")
ax.set_title("Average policy wellbeing statistics by configuration")

ax.set_xticks(x)
ax.set_xticklabels(
    configuration_order,
    rotation=35,
    ha="right",
)

ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()